# 03 · Train (Kaggle GPU, T4 x2)

Runs two experiments in parallel, one per GPU. Checkpoints go to a private HF model repo every 30 min; re-running this notebook with the same experiments resumes where the last session stopped.

**Setup:** Settings → Accelerator: *GPU T4 x2*, Internet: *On*. Add-ons → Secrets: `HF_TOKEN`. Run with *Save Version → Save & Run All*.

Suggested pairs (similar length): `tr000 + tr100`, `tr010 + tr050`, `tr030 + tr_only_control`.

In [ ]:
EXPERIMENTS = ["tr030", "tr_only_control"]  # one per GPU
HF_USER = "DeerKaraca"
DATA_REPO = f"{HF_USER}/bilingual-slm-data"
RUNS_REPO = f"{HF_USER}/bilingual-slm-runs"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/DeerKaraca/bilingual-slm.git
%cd bilingual-slm
!pip install -q -e ".[data,train]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["tokens/*", "tokenizer.json"], local_dir="data")

Optional: measure throughput with a 100-step run (no checkpoints uploaded). Skip once the budget is fixed.

In [ ]:
!python -m slm.train --config configs/experiments/tr050.yaml --set data.tokenizer=data/tokenizer.json \
    out_dir=/kaggle/temp/bench train.total_tokens=26214400 train.eval_every=50

In [ ]:
import subprocess

procs = []
for gpu, name in enumerate(EXPERIMENTS):
    cmd = ["python", "-m", "slm.train", "--config", f"configs/experiments/{name}.yaml",
           "--set", "data.tokenizer=data/tokenizer.json", f"hub_repo={RUNS_REPO}"]
    log = open(f"/kaggle/working/{name}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env))

In [ ]:
import time

# Keep the session alive and print progress until both runs finish.
while any(p.poll() is None for p in procs):
    time.sleep(600)
    for name in EXPERIMENTS:
        print(name, open(f"/kaggle/working/{name}.log").read().strip().splitlines()[-1][:200], flush=True)
print("exit codes:", [p.returncode for p in procs])